# 01 Data Cleaning
Loads the raw Superstore CSV, checks data quality, cleans it, and saves `data/cleaned/superstore_cleaned.csv`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Path is relative to the notebooks/ folder
file_path = "../data/raw/Sample - Superstore.csv"

# This file isn't standard UTF-8, so we give the encoding
df = pd.read_csv(file_path, encoding="windows-1252")

df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
print("Shape (rows, columns):", df.shape)
print()
df.info()

Shape (rows, columns): (9994, 21)

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null

In [4]:
print("Total rows:", len(df))
print("Unique orders:", df["Order ID"].nunique())
print(df[["Sales", "Quantity", "Discount", "Profit"]].dtypes)

Total rows: 9994
Unique orders: 5009
Sales       float64
Quantity      int64
Discount    float64
Profit      float64
dtype: object


In [5]:
df_clean = df.copy()

In [6]:
print(df_clean.isnull().sum())
print()
print("Total missing values:", df_clean.isnull().sum().sum())

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Total missing values: 0


In [7]:
# Row ID is just a counter, so it makes every row look unique.
# We drop it for this check so we can find true duplicates.
dupes = df_clean.drop(columns="Row ID").duplicated()
print("Duplicate rows:", dupes.sum())

Duplicate rows: 1


In [8]:
df_clean[dupes]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3406,3407,US-2014-150119,4/23/2014,4/27/2014,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


In [9]:
cols_without_id = [c for c in df_clean.columns if c != "Row ID"]

df_clean[df_clean.duplicated(subset=cols_without_id, keep=False)]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3405,3406,US-2014-150119,4/23/2014,4/27/2014,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
3406,3407,US-2014-150119,4/23/2014,4/27/2014,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


In [10]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates(subset=cols_without_id, keep="first")

print("Rows before:", before)
print("Rows after:", len(df_clean))
print("Duplicates remaining:", df_clean.duplicated(subset=cols_without_id).sum())

Rows before: 9994
Rows after: 9993
Duplicates remaining: 0


In [11]:
print("Unique orders:", df_clean["Order ID"].nunique())

Unique orders: 5009


In [12]:
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"], format="%m/%d/%Y")
df_clean["Ship Date"] = pd.to_datetime(df_clean["Ship Date"], format="%m/%d/%Y")

print(df_clean[["Order Date", "Ship Date"]].dtypes)
print("Order dates from", df_clean["Order Date"].min(), "to", df_clean["Order Date"].max())

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object
Order dates from 2014-01-03 00:00:00 to 2017-12-30 00:00:00


In [13]:
print("Rows where Ship Date is before Order Date:",
      (df_clean["Ship Date"] < df_clean["Order Date"]).sum())

Rows where Ship Date is before Order Date: 0


In [14]:
df_clean["Postal Code"] = df_clean["Postal Code"].astype(str).str.zfill(5)
print(df_clean["Postal Code"].head())

0    42420
1    42420
2    90036
3    33311
4    33311
Name: Postal Code, dtype: str


In [15]:
text_cols = ["Ship Mode", "Segment", "Country", "Region", "Category", "Sub-Category"]

for col in text_cols:
    print(col, "->", sorted(df_clean[col].unique()))
    print()

Ship Mode -> ['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment -> ['Consumer', 'Corporate', 'Home Office']

Country -> ['United States']

Region -> ['Central', 'East', 'South', 'West']

Category -> ['Furniture', 'Office Supplies', 'Technology']

Sub-Category -> ['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']



In [16]:
all_text_cols = ["Order ID", "Ship Mode", "Customer ID", "Customer Name", "Segment",
                 "Country", "City", "State", "Region", "Product ID",
                 "Category", "Sub-Category", "Product Name"]

for col in all_text_cols:
    df_clean[col] = df_clean[col].str.strip()

In [17]:
print(df_clean[["Sales", "Quantity", "Discount", "Profit"]].describe())
print()
print("Sales <= 0:", (df_clean["Sales"] <= 0).sum())
print("Quantity <= 0:", (df_clean["Quantity"] <= 0).sum())
print("Discount outside 0 to 1:", ((df_clean["Discount"] < 0) | (df_clean["Discount"] > 1)).sum())
print("Rows with negative Profit:", (df_clean["Profit"] < 0).sum())

              Sales     Quantity     Discount       Profit
count   9993.000000  9993.000000  9993.000000  9993.000000
mean     229.852846     3.789753     0.156188    28.660971
std      623.276074     2.225149     0.206457   234.271476
min        0.444000     1.000000     0.000000 -6599.978000
25%       17.280000     2.000000     0.000000     1.731000
50%       54.480000     3.000000     0.200000     8.671000
75%      209.940000     5.000000     0.200000    29.364000
max    22638.480000    14.000000     0.800000  8399.976000

Sales <= 0: 0
Quantity <= 0: 0
Discount outside 0 to 1: 0
Rows with negative Profit: 1870


In [18]:
df_clean.to_csv("../data/cleaned/superstore_cleaned.csv", index=False)
print("Saved. Final shape:", df_clean.shape)

Saved. Final shape: (9993, 21)


In [19]:
check = pd.read_csv("../data/cleaned/superstore_cleaned.csv")
print("Saved file shape:", check.shape)

Saved file shape: (9993, 21)


## Data-quality log

| Check | Result | Action |
|---|---|---|
| Rows loaded | 9,994 | None |
| Missing values | 0 | None |
| Exact duplicates (ignoring Row ID) | 1 | Removed, leaving 9,993 rows |
| Ship Date before Order Date | 0 | None |
| Invalid Sales / Quantity / Discount | 0 | None |
| Negative-profit rows | 1,870 | Kept on purpose (real losses) |
| Order dates | 2014-01-03 to 2017-12-30 | None |
| Country column | One value only | Left in, not used |